In [119]:
import boto3
import json

In [120]:
client = boto3.client("bedrock-runtime", region_name="eu-central-1")
# Use Haiku for faster evals
model_id = "eu.anthropic.claude-haiku-4-5-20251001-v1:0"


def add_user_message(messages, text):
    user_message = {"role": "user", "content": [{"text": text}]}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": [{"text": text}]}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {
            "temperature": temperature,
            "stopSequences": stop_sequences,
        },
    }

    if system:
        params["system"] = [{"text": system}]

    response = client.converse(**params)

    return response["output"]["message"]["content"][0]["text"]

In [121]:
def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluting the solution",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)


In [122]:
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)


In [123]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the results"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [124]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Solution criteria to Evaluate:
<solution_criteria>
{test_case["solution_criteria"]}
</solution_criteria>


Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [125]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [126]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "Output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [127]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [128]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)


Average score: 8.333333333333334


In [129]:
print(json.dumps(results, indent=2))

[
  {
    "Output": "\n{\n  \"FunctionName\": \"ProcessS3Events\",\n  \"Runtime\": \"python3.11\",\n  \"Role\": \"arn:aws:iam::ACCOUNT_ID:role/lambda-s3-execution-role\",\n  \"Handler\": \"index.handler\",\n  \"Timeout\": 300,\n  \"MemorySize\": 512,\n  \"Description\": \"Lambda function that processes S3 events\",\n  \"Environment\": {\n    \"Variables\": {\n      \"BUCKET_NAME\": \"my-s3-bucket\"\n    }\n  },\n  \"Events\": {\n    \"S3Event\": {\n      \"Type\": \"S3\",\n      \"Properties\": {\n        \"Bucket\": \"my-s3-bucket\",\n        \"Events\": [\n          \"s3:ObjectCreated:*\",\n          \"s3:ObjectRemoved:*\"\n        ]\n      }\n    }\n  },\n  \"Architectures\": [\n    \"x86_64\"\n  ],\n  \"EphemeralStorage\": {\n    \"Size\": 512\n  },\n  \"Layers\": [],\n  \"VpcConfig\": {\n    \"SecurityGroupIds\": [],\n    \"SubnetIds\": []\n  },\n  \"Tags\": {\n    \"Environment\": \"production\",\n    \"Purpose\": \"s3-event-processing\"\n  }\n}\n",
    "test_case": {
      "task